# Lezione 3: salva il tuo modello, scala i tuoi dati e interpreta la regressione lineare

¿Alguna vez cerraste un videojuego a mitad de partida y, al volver a abrirlo, seguía exactamente donde lo dejaste? Eso pasa grazie a que el juego guarda su estado en un archivo. Hoy vas a hacer lo mismo, pero con un modelo de aprendizaje automático (ML, del inglés Machine Learning).

Nella lezione precedente entrenaste tu primer modelo de regresión lineal con una serie di prezzi simulada. Funcionó, pero seguro que notaste un problema: cada vez que quieres usar el modelo, tienes que volver a entrenarlo desde cero. Si tu insieme di dati es pequeño, apenas se nota. Si algún día trabajas con miles o millones de filas, reentrenar cada vez que necesitas una previsione puede tardar minutos u horas.

En esta clase resuelves tres cuestiones muy prácticas:

- Cómo guardar un modelo ya entrenado para reutilizarlo sin repetir el entrenamiento
- Cómo preparar tus datos cuando las columnas tienen escalas muy distintas entre sí
- Cómo leer con criterio los números que scikit-learn te entrega tras entrenar: la pendiente, el punto de corte y la puntuación de ajuste

Todo el código de este cuaderno funciona directamente en Google Colab, sin conexiones externas: solo necesitas NumPy y scikit-learn, que ya vienen instalados.

## Guarda tu modelo entrenado con pickle

Piensa otra vez en el videojuego. Si cada vez que lo abrieras tuvieras que empezar desde el primer nivel, ¿cuánta gente seguiría jugando? Por eso existe la función de guardar partida: capturas el estado exacto del juego en un archivo y, más tarde, lo recuperas justo donde lo dejaste.

Un modelo de ML funciona igual. Entrenarlo es como avanzar niveles: consume tiempo y recursos. Una vez que el modelo "aprendió" sus parámetros a partire da los datos, quieres guardar ese estado para no repetir el proceso cada vez que abras tu programa.

En Python, este mecanismo se llama pickling. El módulo `pickle`, incluido en la instalación estándar del lenguaje, convierte casi cualquier objeto (un modelo, una lista, un diccionario) en una secuencia de bytes que puedes escribir en un archivo. Después, con la operación inversa, reconstruyes el objeto original tal como estaba, sin volver a calcular nada.

Vamos a retomar la serie di prezzi simulada de la clase anterior y a entrenar el modelo.

In [ ]:
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression

# Simulamos de nuevo una serie de precios con un camino aleatorio (random walk)
np.random.seed(42)
dias = np.arange(100).reshape(-1, 1)          # variable independiente: el día
pasos = np.random.normal(loc=0, scale=1, size=100)
precios = 100 + np.cumsum(pasos)              # variable dependiente: el precio simulado

# Entrenamos el modelo, igual que en la clase anterior
modelo = LinearRegression()
modelo.fit(dias, precios)

print("Modelo entrenado.")
print("Coeficiente (pendiente):", modelo.coef_[0])

Ora salva quel modello già addestrato in un file. Nota due dettagli del codice seguente. Primo, apri il file in modalità `"wb"` (write binary, scrittura binaria), perché il contenuto non è testo leggibile, ma byte. Secondo, per convenzione, i file generati con `pickle` hanno estensione `.pkl`, anche se Python non ti obbliga a usarla.

In [ ]:
# Guardamos el modelo entrenado en un archivo binario
with open("modelo_precios.pkl", "wb") as archivo:
    pickle.dump(modelo, archivo)

print("Modelo guardado en modelo_precios.pkl")

### Simula una nuova sessione di lavoro

Ora arriva la parte interessante. Simuliamo che chiudi il programma, spegni il computer e torni un altro giorno. Ti serve il codice di addestramento per continuare a usare il modello? No, ti serve solo il file salvato.

Per simularlo davvero in Colab, potresti andare nel menu "Ambiente di esecuzione" e scegliere "Riavvia ambiente di esecuzione". Qui lo simuliamo in modo più rapido: eliminiamo la variabile `modelo` dalla memoria con `del` e dimostriamo che, anche così, puoi recuperare il modello completo dal file.

In [ ]:
# Simulamos el cierre del programa: borramos el modelo de la memoria
del modelo

# A partir de aquí, "olvidamos" que alguna vez existió la variable "modelo"
import pickle

# Cargamos el modelo ya entrenado, sin volver a entrenarlo
with open("modelo_precios.pkl", "rb") as archivo:
    modelo_cargado = pickle.load(archivo)

# Lo usamos directamente para predecir el precio del día 150
prediccion = modelo_cargado.predict([[150]])
print(f"Precio estimado para el día 150: {prediccion[0]:.2f}")

Esta vez abres el archivo en modo `"rb"` (read binary, lectura binaria). El modelo cargado es funcionalmente idéntico al original: mismos coeficientes, mismas predicciones, cero minutos de entrenamiento adicional.

### Una alternativa moderna: joblib

Existe una alternativa a `pickle` que muchos proyectos de ML prefieren: la librería `joblib`. Su sintaxis es casi idéntica y funciona meglio con modelos que almacenan grandes arreglos numéricos internamente, algo usuale en scikit-learn.

In [ ]:
import joblib

# Guardar con joblib
joblib.dump(modelo_cargado, "modelo_precios.joblib")

# Cargar con joblib
modelo_joblib = joblib.load("modelo_precios.joblib")

print("Predicción con el modelo cargado vía joblib:", modelo_joblib.predict([[150]]))

Una advertencia importante: un modelo guardado no aprende nada nuevo por sí solo. Si los datos del mundo real cambian con el tiempo, tu modelo guardado sigue reflejando patrones antiguos. Por eso conviene establecer un calendario de reentrenamiento, por esempio una vez al mes, en lugar de asumir que un modelo pickled sirve para siempre.

**Guarda tu propio modelo ahora mismo** ejecutando de nuevo las celdas anteriores y comprueba que el archivo `modelo_precios.pkl` aparece en el panel de archivos de Colab (icono de carpeta en el menú lateral).

## Por qué a veces hay que escalar los datos

Imagina que vuoi prevedere el precio de una vivienda a partire da dos caratteristiche: su superficie en metros cuadrados (valores entre 40 y 300, aproximadamente) y su antigüedad en años (valores entre 0 y 50). Estas dos columnas viven en escalas completamente distintas. ¿Qué pasa si un algoritmo mezcla números que van de 0 a 50 con otros que van de 40 a 300? Para muchos algoritmos, esa diferencia de magnitud no es un simple detalle estético: puede hacer que la característica con números más grandes domine el aprendizaje, sebbene non sea la más relevante para el problema.

El escalado consiste en transformar cada columna affinché tutte queden en rangos comparables, sin alterar la relación relativa entre sus valores. La técnica más usada se llama estandarización: resta la media de cada columna y divide entre su desviación estándar, de modo que el resultado tiene media cero y varianza uno. Scikit-learn la implementa en la clase `StandardScaler`.

Vamos a comprobarlo en la práctica. Ampliamos el esempio con una segunda característica simulada, el volumen de operaciones de cada día, que se mueve en una escala mucho mayor que el propio día.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)
dias = np.arange(100)
volumen = np.random.normal(loc=50000, scale=8000, size=100)  # escala grande
precios = 100 + np.cumsum(np.random.normal(0, 1, size=100))

# Combinamos ambas características en una sola matriz
X = np.column_stack([dias, volumen])
y = precios

# Antes de escalar: mira la diferencia de magnitud entre columnas
print("Sin escalar, primera fila:", X[0])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ajustamos el escalador SOLO con los datos de entrenamiento
escalador = StandardScaler()
X_train_escalado = escalador.fit_transform(X_train)

# Aplicamos la MISMA transformación aprendida a los datos de prueba
X_test_escalado = escalador.transform(X_test)

print("Escalado, primera fila:", X_train_escalado[0])

Nota un dettaglio che segna la differenza tra un flusso corretto e uno con fughe di informazione: adatti (`fit_transform`) lo scaler solo sui dati di addestramento, e poi applichi (`transform`, senza `fit`) quella stessa trasformazione ai dati di test. Se adattassi lo scaler su tutto l'insieme, il tuo modello avrebbe accesso indiretto a informazioni dei dati di test prima della valutazione, e questo falserebbe i tuoi risultati.

Confrontiamo ora, con numeri concreti, come cambiano la media e la deviazione standard di ogni colonna prima e dopo lo scaling.

In [ ]:
print("--- Antes de escalar ---")
print("Media por columna:", X_train.mean(axis=0))
print("Desviación estándar por columna:", X_train.std(axis=0))

print("\n--- Después de escalar ---")
print("Media por columna:", X_train_escalado.mean(axis=0).round(2))
print("Desviación estándar por columna:", X_train_escalado.std(axis=0).round(2))

Antes de escalar, la columna del volumen tiene valores de decenas de miles mentre la del día apenas llega a un centenar. Después de escalar, ambas columnas quedan alrededor de cero, con una dispersión similar (varianza uno).

Para la regressione lineal simple con una sola característica, como la de la clase anterior, el escalado no cambia demasiado la calidad de las predicciones. Su verdadero valor aparece con varias caratteristiche de escalas distintas: meglioa la interpretación de los coeficientes, porque quedan en unidades comparables, y ayuda a que algoritmos de optimización como el descenso de gradiente converjan más rápido.

**Aplica `StandardScaler` a tus propios datos** antes de entrenar cualquier modelo con más de una característica.

## Qué significan de verdad los coeficientes, el intercepto y el R²

Cuando entrenas un modelo de regresión lineal con scikit-learn, tres atributos concentran toda la información que necesitas para interpretarlo: `coef_`, `intercept_` y el resultado de `score()`. ¿Alguna vez viste estos números en pantalla y te preguntaste qué significan davvero? Vale la pena entenderlo más allá del número.

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

np.random.seed(42)
dias = np.arange(100).reshape(-1, 1)
precios = 100 + np.cumsum(np.random.normal(0, 1, size=100))

modelo_simple = LinearRegression()
modelo_simple.fit(dias, precios)

print("Pendiente (coef_):", modelo_simple.coef_)
print("Intercepto (intercept_):", modelo_simple.intercept_)
print("R² sobre los mismos datos (score()):", modelo_simple.score(dias, precios))

**El coeficiente (`coef_`), o pendiente.** Indica cuánto cambia la previsione por cada unidad adicional de la variable independiente. Si `coef_` vale 0,8, cada día adicional añade, en promedio, 0,8 unidades de precio según el patrón aprendido. Un coeficiente positivo señala una relación creciente; uno negativo, una relación decreciente. Cuando tienes varias caratteristiche, cada una recibe su propio coeficiente, y ahí es donde el escalado che hai appena ver facilita il confronto cuál pesa más.

**El intercepto (`intercept_`).** Es el valor que prevede il modello cuando todas las variables independientes valen cero. En nuestro esempio, sería el precio estimado para el día cero. A veces este valor tiene sentido real; otras veces es solo un punto matemático necesario para definir la recta, sin interpretación práctica directa (por esempio, si "antigüedad de la vivienda igual a cero" no aparece nunca en tus datos reales). Interprétalo con cautela cuando quede fuera del rango de tus datos originales.

**El coefficiente di determinazione (R²), obtenido con `score()`.** Mide qué proporción de la variabilidad de tus datos explica el modelo, en una escala que normalmente va de 0 a 1 (puede volverse negativo si el modelo predice peor que simplemente usar la media). Un R² de 0,95 significa che el modelo explica el 95% de la variación observada en tus datos: un ajuste muy bueno. Un R² de 0,3 significa che el modelo solo explica el 30%: la recta capta algo de la tendencia, pero deja fuera la mayor parte del comportamiento de los datos, probablemente porque faltan variables relevantes o porque la relación no es lineal.

¿Un R² alto garantiza siempre un buen modelo? No necesariamente, y uno bajo tampoco lo descarta siempre: depende del problema. Predecir el comportamiento humano o los mercados financieros con un R² de 0,3 puede ser un resultado aceptable, mentre un R² de 0,3 en un proceso físico controlado suele indicar que falta algo importante en tu planteamiento.

## Riassunto: lo que te llevas de esta clase

- El módulo `pickle` serializa un modelo entrenado en un archivo, así evitas reentrenar cada vez que lo necesitas
- `pickle.dump()` guarda el objeto en modo escritura binaria (`"wb"`); `pickle.load()` lo recupera en modo lectura binaria (`"rb"`)
- `joblib` es una alternativa moderna a `pickle`, recomendable para modelos de scikit-learn con arreglos numéricos grandes
- `StandardScaler` transforma tus columnas a media cero y varianza uno, evitando que una característica con escala mayor domine el aprendizaje
- Ajusta el escalador solo con los datos de entrenamiento y aplícalo después a los de prueba, nunca al revés
- `coef_`, `intercept_` y `score()` te dicen, respectivamente, cuánto influye cada variable, dónde arranca la recta y qué porcentaje de la variación explica tu modelo

**Ahora te toca a ti.** Abre el cuaderno de práctica de esta clase y pon a prueba lo che hai appena aprender con tus propios datos simulados.

**Siglas usadas en esta clase:** ML (Machine Learning), R² (coefficiente di determinazione).